# Fine-tune Whisper on a South African language — local RTX 3060

Tuned for a single local GPU (RTX 3060, 12GB). `whisper-small` fits comfortably;
`whisper-medium` is tight but workable with small batches + gradient checkpointing.
`whisper-large-v3` is not realistic on 12GB for fine-tuning — stick to small/medium locally.

**Expected data layout**: two CSVs, `train.csv` and `eval.csv`, each with columns:
- `audio_path` — path to a 16kHz mono `.wav` file
- `sentence` — transcript, natural casing/punctuation (don't strip it — Whisper models it)

## 1. Install dependencies

In [1]:
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa tensorboard

## 2. Check GPU

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM (GB): 12.9


## 3. Config — edit these

In [3]:
LANGUAGE_TOKEN = "sw"  # closest Whisper-supported language code (Swahili as Bantu anchor if
                        # your target language isn't natively supported — confirm this fits your case)
TASK = "transcribe"
BASE_MODEL = "openai/whisper-small"  # use "openai/whisper-medium" only if VRAM allows
OUTPUT_DIR = "./whisper-target-lang"
TRAIN_CSV = "train.csv"
EVAL_CSV = "eval.csv"

# RTX 3060 12GB with whisper-small: batch 8-16 is workable.
# Drop to 4-8 for whisper-medium, and keep gradient_checkpointing on.
PER_DEVICE_TRAIN_BATCH = 8
PER_DEVICE_EVAL_BATCH = 4
GRAD_ACCUM_STEPS = 2

## 4. Imports

In [4]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Audio, Dataset
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

## 5. Load data

In [5]:
LANGUAGE = 'tsn'

dataset_dict = load_dataset(
    "dsfsi-anv/za-african-next-voices-compressed",
    LANGUAGE,
)

dataset_dict["train"] = dataset_dict["train"].select(range(2500))
dataset_dict["dev"] = dataset_dict["dev"].select(range(250))
dataset_dict["dev_test"] = dataset_dict["dev_test"].select(range(10))

dataset_dict = dataset_dict.cast_column("audio", Audio(sampling_rate=16000))

In [6]:
for split in ["train", "dev"]:
    print(split)
    dur = sum(dataset_dict[split]["duration"])
    print(f"total {dur} in seconds")
    print(f"total {dur/3600} in hours")
    print()

train
total 44491.19281499991 in seconds
total 12.358664670833308 in hours

dev
total 7738.436803 in seconds
total 2.149565778611111 in hours



In [7]:
def is_valid_transcript(batch):
    t = batch["transcript"]
    return t is not None and isinstance(t, str) and t.strip() != ""

In [8]:
import re


def normalize_text(batch):
    if batch["transcript"] is None:
        return batch

    text = batch["transcript"]

    # Remove annotation tags like [pause], [cs], [?], [noise] etc.
    text = re.sub(r'\[.*?\]', '', text)

    # Collapse repeated whitespace and trim ends
    text = re.sub(r'\s+', ' ', text).strip()

    batch["transcript"] = text
    return batch

In [9]:
for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=4)
    dataset_dict[split] = dataset_dict[split].map(normalize_text)

## 6. Load feature extractor, tokenizer, processor

In [10]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(BASE_MODEL)
tokenizer = WhisperTokenizer.from_pretrained(BASE_MODEL, language=LANGUAGE_TOKEN, task=TASK)
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language=LANGUAGE_TOKEN, task=TASK)

## 7. Preprocess audio + labels

This reads every wav file — the slowest step locally. Bump `num_proc` if you have spare CPU cores.

In [11]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["transcript"]).input_ids
    return batch



In [12]:
for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=4)
    dataset_dict[split] = dataset_dict[split].map(
        prepare_dataset,
        remove_columns=dataset_dict[split].column_names,
        num_proc=2,  # bump to e.g. 4 if you have CPU cores to spare
    )

Map (num_proc=2):   0%|          | 0/2498 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/250 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/250 [00:00<?, ? examples/s]

## 8. Data collator

In [13]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 9. Metrics (WER / CER)

In [14]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {
        "wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": 100 * cer_metric.compute(predictions=pred_str, references=label_str),
    }

## 10. Load model

In [15]:
import torch

model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.language = LANGUAGE_TOKEN
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

## 11. Training arguments

`fp16` + `gradient_checkpointing` are what make whisper-small/medium fit in 12GB — keep both on.

In [16]:
import math

def create_training_args(
    output_dir,
    train_dataset,
    epochs,
    per_device_train_batch_size,
    per_device_eval_batch_size,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_ratio=0.3,
    evals_per_epoch=4,
    logs_per_epoch=10,
):
    """
    Creates Seq2SeqTrainingArguments with step counts computed from epochs.

    Args:
        train_dataset: Hugging Face training dataset.
        epochs: Number of training epochs.
        warmup_ratio: Fraction of total steps used for warmup.
        evals_per_epoch: Number of evaluations per epoch.
        logs_per_epoch: Number of logging events per epoch.
    """

    world_size = max(torch.cuda.device_count(), 1)

    steps_per_epoch = math.ceil(
        len(train_dataset)
        / (
            per_device_train_batch_size
            * gradient_accumulation_steps
            * world_size
        )
    )

    max_steps = steps_per_epoch * epochs
    warmup_steps = max(1, int(max_steps * warmup_ratio))
    eval_steps = max(1, steps_per_epoch // evals_per_epoch)
    logging_steps = max(1, steps_per_epoch // logs_per_epoch)

    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        save_strategy="no",
        load_best_model_at_end=False,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        gradient_checkpointing=True,
        fp16=torch.cuda.is_available(),
        eval_strategy="steps",
        predict_with_generate=True,
        generation_max_length=225,
        save_steps=max_steps + 1,  # disables checkpointing when save_strategy="no"
        eval_steps=eval_steps,
        logging_steps=logging_steps,
        report_to=[],
        metric_for_best_model="wer",
        greater_is_better=False,
        push_to_hub=False,
    )

In [25]:
# training_args = Seq2SeqTrainingArguments(
#     output_dir=OUTPUT_DIR,
#     save_strategy="no",
#     load_best_model_at_end=False, #do not load best model at end if you want to save checkpoints
#     per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
#     per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
#     gradient_accumulation_steps=GRAD_ACCUM_STEPS,
#     learning_rate=1e-5,
#     warmup_steps=5,
#     max_steps=40,
#     gradient_checkpointing=True,
#     fp16=torch.cuda.is_available(),
#     eval_strategy="steps",
#     predict_with_generate=True,
#     generation_max_length=225,
#     save_steps=50,
#     eval_steps=10,
#     logging_steps=10,
#     report_to=[],#["tensorboard"],
#     metric_for_best_model="wer",
#     greater_is_better=False,
#     push_to_hub=False,
# )

training_args = create_training_args(
    output_dir=OUTPUT_DIR,
    train_dataset=dataset_dict["train"],
    epochs=3,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

## 12. Train

Seq2seq generation during eval is slower than CTC — expect eval steps to take noticeably longer than train steps. Keep an eye on `nvidia-smi` for VRAM headroom, especially if you bump to whisper-medium.

In [26]:
trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
39,2.346699,1.364842,75.042563,44.354067
78,2.115312,1.317207,75.723572,45.906916
117,2.026199,1.250469,62.343291,39.292446
156,1.874094,1.128846,64.626219,40.287081
195,1.294278,1.098942,68.394985,42.008119
234,1.486855,1.058951,68.681319,42.113963
273,1.290790,1.040000,65.763814,41.345513
312,1.327580,1.023074,59.596038,38.123822
351,1.108120,1.023765,64.401795,42.329999
390,1.103448,1.017532,68.170562,44.430912


TrainOutput(global_step=471, training_loss=1.4605325356663665, metrics={'train_runtime': 5739.3205, 'train_samples_per_second': 1.313, 'train_steps_per_second': 0.082, 'total_flos': 2.16265898999808e+18, 'train_loss': 1.4605325356663665, 'epoch': 3.0})

## 13. Save

In [27]:
# trainer.save_model(OUTPUT_DIR)
# processor.save_pretrained(OUTPUT_DIR)
# print(f"Saved to {OUTPUT_DIR}")

## 14. Quick sanity-check inference

In [28]:
import soundfile as sf

# example:
# speech, sr = sf.read("some_eval_clip.wav")
# inputs = processor(speech, sampling_rate=16000, return_tensors="pt").input_features.to(model.device)
# with torch.no_grad():
#     predicted_ids = model.generate(inputs)
# print(processor.batch_decode(predicted_ids, skip_special_tokens=True))

In [29]:
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
 

In [30]:
# sample = dataset_dict["dev"].select(range(1))[0]
# input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

# with torch.no_grad():
#     logits = model(input_values).logits

# predicted_ids = torch.argmax(logits, dim=-1)
# print("Raw predicted IDs:", predicted_ids[0].tolist())
# print("Unique IDs predicted:", set(predicted_ids[0].tolist()))
# print("Pad/blank token ID:", processor.tokenizer.pad_token_id)

# Test 1: preprocessed dev split

In [31]:
def test_on_eval_set(num_samples=5):
    print("=== Evaluation on dev split ===\n")
    test_samples = dataset_dict["dev"].select(range(num_samples))

    predictions = []
    references = []

    for i, sample in enumerate(test_samples):
        input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to(device)

        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                max_new_tokens=225,   # cap output length
                language="en",        # set to your target language code, or omit to let Whisper auto-detect
                task="transcribe",
            )

        predicted_text = processor.tokenizer.decode(predicted_ids[0], skip_special_tokens=True)

        # sample["labels"] has -100 padding — swap back to pad_token_id before decoding
        label_ids = [t if t != -100 else processor.tokenizer.pad_token_id for t in sample["labels"]]
        actual_text = processor.tokenizer.decode(label_ids, skip_special_tokens=True)

        predictions.append(predicted_text)
        references.append(actual_text)

        print(f"--- Sample {i+1} ---")
        print(f"Predicted: {predicted_text}")
        print(f"Actual:    {actual_text}\n")

    wer = wer_metric.compute(predictions=predictions, references=references)
    cer = cer_metric.compute(predictions=predictions, references=references)
    print(f"WER: {wer:.4f}")
    print(f"CER: {cer:.4f}")

    return predictions, references

In [32]:
test_on_eval_set(num_samples=5)

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Evaluation on dev split ===



[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Sample 1 ---
Predicted:  Go duela legetho go tlhokiwa ditlhang ka natsedingwe morago ga ngwaga ya ditšhelete mme ga di ikgoni go dirisiwa go laola ditšhelete tsa go tsametsa kgwebo ya gago.
Actual:    Go duela lekgetho go tlhokiwa ditlhankana tse dingwe morago ga ngwaga ya ditšhelete mme ga di kgonwe go dirisiwa go laola ditšhelete tsa go tsamaisa kgwebo ya gago.



[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Sample 3 ---
Predicted:  Risaeke ya kwele o tlase ya thobo ya dijalo.
Actual:    Riseke ya kwelotlase ya thobo ya dijalo



[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Sample 4 ---
Predicted:  The next generation of the Tswakilweng is a new generation of science and science.
Actual:    Direšene tse di tswakilweng sekgwebo di siame thata e bile di lekalekana sesaense, fela di a tura.

--- Sample 5 ---
Predicted:  Go na go tshweefatsa pelo go utlwa gore balemirui ba bantsi ba le bogabasadi, ba role ba roadi ba bona ba ba ipofilwang go ba thusa go aga tsamaiso ya bolemirui ka go bantsi nna ya thuso mo masemong, go laola tsa mo ofising le ka go thusa go makgetha le go rekisa kgomo ya bona.
Actual:    Go ne go tshweufatsa pelo go utlwa gore balemirui ba bantsi ba leboga basadi, barwa le barwadi ba bona ba ba ipofileng go ba thusa go aga tsamaiso ya bolemirui ka go ba naya thuso mo masimong, go laola tsa mo ofising le ka go thusa go makheta le go rekisa khumo ya bona.

WER: 0.5263
CER: 0.3088


([' Go duela legetho go tlhokiwa ditlhang ka natsedingwe morago ga ngwaga ya ditšhelete mme ga di ikgoni go dirisiwa go laola ditšhelete tsa go tsametsa kgwebo ya gago.',
  ' The first one is the one that has been used in the past for a long time.',
  ' Risaeke ya kwele o tlase ya thobo ya dijalo.',
  ' The next generation of the Tswakilweng is a new generation of science and science.',
  ' Go na go tshweefatsa pelo go utlwa gore balemirui ba bantsi ba le bogabasadi, ba role ba roadi ba bona ba ba ipofilwang go ba thusa go aga tsamaiso ya bolemirui ka go bantsi nna ya thuso mo masemong, go laola tsa mo ofising le ka go thusa go makgetha le go rekisa kgomo ya bona.'],
 ['Go duela lekgetho go tlhokiwa ditlhankana tse dingwe morago ga ngwaga ya ditšhelete mme ga di kgonwe go dirisiwa go laola ditšhelete tsa go tsamaisa kgwebo ya gago.',
  'Phuthego e ne e beilwe go keteka nako le matsapa a setlhopha se a dirisitseng go tsweletse pele lenaneo la go bona balemirui ba ba ka tlhabololwang.',
